# qwen_ft GenImage Zip Export on Kaggle (Modal path)

This notebook is a **thin wrapper** for the Modal fine-tuning flow
(`docs/runbook-qwen-finetune-modal.md`): it verifies the GenImage data root,
exports a balanced protocol-relative zip archive, and prints the archive to
download. No GCS bucket, no Vertex endpoint, and no GCP service-account
secret are required.

- Protocol: `protocol-a-small`, 100 train and 50 eval images per label per
  generator, seed 70.
- Output: `/kaggle/working/qwen_ft_protocol_a_small.zip` — save the notebook
  output and download this file, then upload it to Modal Volume
  `aiforensics-qwen-ft` at `/archives/`.
- Enable **Internet** in the Kaggle notebook settings.

## 1. Install the repository and dependencies

The notebook clones this repository into writable storage and installs the
package. Nothing is pinned here; dependency names come from `pyproject.toml`.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/ai-image-forensics")
REPO_GIT_URL = "https://github.com/Nnguyen-dev2805/ai-image-forensics.git"

if not REPO_ROOT.exists():
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_GIT_URL, str(REPO_ROOT)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-e", "."],
    cwd=str(REPO_ROOT),
    check=True,
)
print("repository:", REPO_ROOT)

## 3. Verify the GenImage data root

The export expects the Kaggle dataset layout
`<DATA_ROOT>/<generator>/<train|val>/{ai,nature}/*`. The cell lists the
generator directories it can see; the export script fails loudly when a
configured generator is missing or has too few images.

In [ ]:
DATA_ROOT = Path("/kaggle/input/datasets/yangsangtai/tiny-genimage")

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"DATA_ROOT does not exist: {DATA_ROOT}. Attach the GenImage dataset "
        "and point DATA_ROOT at its directory."
    )

split_dirs = ("train", "val")
found = [
    entry.name
    for entry in sorted(DATA_ROOT.iterdir())
    if entry.is_dir() and any((entry / split).is_dir() for split in split_dirs)
]
print("generator directories:", len(found))
for name in found:
    print("  -", name)
if not found:
    raise FileNotFoundError(
        f"No <generator>/<split> layout under {DATA_ROOT}; check the attachment."
    )

## 3. Export the protocol zip archive

In [ ]:
# AIF_SECTION: modal_zip_export
import shlex
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/yangsangtai/tiny-genimage")
ARCHIVE_PATH = Path("/kaggle/working/qwen_ft_protocol_a_small.zip")

EXPORT_CMD = (
    f"{sys.executable} scripts/export_genimage_qwen_ft_subset.py "
    f"--data-root {DATA_ROOT} "
    "--protocol protocol-a-small "
    "--train-per-label 100 "
    "--eval-per-label 50 "
    "--seed 70 "
    "--path-mode modal-volume "
    f"--archive-path {ARCHIVE_PATH}"
)

subprocess.run(shlex.split(EXPORT_CMD), cwd=str(REPO_ROOT), check=True)

import zipfile

with zipfile.ZipFile(ARCHIVE_PATH) as zf:
    names = zf.namelist()
print("archive:", ARCHIVE_PATH, ARCHIVE_PATH.stat().st_size, "bytes")
print("zip entries:", len(names))
print("next step: download this file and run")
print("  python3 -m modal volume put aiforensics-qwen-ft "
      f"{ARCHIVE_PATH.name} /archives/{ARCHIVE_PATH.name}")